In [1]:
import numpy as np
import pandas as pd
import joblib
from keras.models import load_model
from sklearn.metrics import mean_absolute_error

SEQ_LEN = 24

model = load_model("energy_model_final.keras")
scaler_x = joblib.load("scaler_x.pkl")
scaler_y = joblib.load("scaler_y.pkl")

FEATURES = list(scaler_x.feature_names_in_)

df = pd.read_csv("consumptions.csv")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values("Timestamp").reset_index(drop=True)

#log transform (CRITICAL)
df["Energy_log"] = np.log1p(df["EnergyConsumption"])

#time
df["year"] = df["Timestamp"].dt.year
df["month"] = df["Timestamp"].dt.month
df["dayofyear"] = df["Timestamp"].dt.dayofyear

df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)

df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

weekday_map = {
"Monday":0,"Tuesday":1,"Wednesday":2,
"Thursday":3,"Friday":4,"Saturday":5,"Sunday":6
}

df["weekday_num"] = df["weekday"].map(weekday_map)

hour_week = df["hour"] + df["weekday_num"]*24
df["hour_week_sin"] = np.sin(2*np.pi*hour_week/168)
df["hour_week_cos"] = np.cos(2*np.pi*hour_week/168)

#weather
df["temp_sq"] = df["tavg"]**2
df["temp_humidity"] = df["tavg"]*df["humidity"]
df["temp_wind"] = df["tavg"]*df["wspd"]
df["wind_humidity"] = df["wspd"]*df["humidity"]

#lag (from LOG energy)
lags = [1,2,3,6,12,24,48,72,168,336]

for l in lags:
    df[f"lag_{l}"] = df["Energy_log"].shift(l)

#diff
df["diff_1"] = df["Energy_log"].diff(1)
df["diff_24"] = df["Energy_log"].diff(24)

#rolling(shift first like training)
shifted = df["Energy_log"].shift(1)

windows = [6,12,24,48,168]

for w in windows:
    df[f"roll_mean_{w}"] = shifted.rolling(w).mean()
    df[f"roll_std_{w}"]  = shifted.rolling(w).std()

for w in [6,24,168]:
    df[f"roll_min_{w}"] = shifted.rolling(w).min()
    df[f"roll_max_{w}"] = shifted.rolling(w).max()


df["city_Izmir"] = 1

# weekday encoding
for i in range(7):
    df[f"dow_{i}"] = (df["weekday_num"]==i).astype(int)

df = df.dropna().reset_index(drop=True)

# matrix
X = scaler_x.transform(df[FEATURES])


X_seq=[]
y_true=[]

for i in range(SEQ_LEN,len(X)):
    X_seq.append(X[i-SEQ_LEN:i])
    y_true.append(df["EnergyConsumption"].iloc[i])

X_seq=np.array(X_seq)
y_true=np.array(y_true)

#predict
pred_scaled=model.predict(X_seq)

pred_log=scaler_y.inverse_transform(pred_scaled)
pred=np.expm1(pred_log).flatten()

#metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- Core metrics ---
mae = mean_absolute_error(y_true, pred)
rmse = np.sqrt(mean_squared_error(y_true, pred))
mape = np.mean(np.abs((y_true - pred) / (y_true + 1e-8))) * 100
r2 = r2_score(y_true, pred)

# --- Additional diagnostics ---
bias = np.mean(pred - y_true)

# Peak error (error at highest true demand)
peak_idx = np.argmax(y_true)
peak_error = pred[peak_idx] - y_true[peak_idx]

# --- Print nicely ---
print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("MAPE:", round(mape, 2), "%")
print("R2:", round(r2, 4))
print("Bias:", round(bias, 4))
print("Peak Error:", round(peak_error, 4))

252/252 [==============================] - 2s 6ms/step
MAE: 543.0886
RMSE: 839.0493
MAPE: 1.35 %
R2: 0.984
Bias: -13.8091
Peak Error: -1356.4903
